# ML-09 — Validation and Claim Audit

This notebook checks paper claims, client-grouped validation, and feature leakage.


## 1. Two paper findings and methodology questions

On page 8 of the March 2026 FlyRank research paper, click-through rate falls across worse average-position tiers. It is a weighted portfolio comparison, so the useful question is whether the same pattern appears in this warehouse slice when total clicks are divided by total impressions in each tier. It is not a label or an intervention result.

Page 29 reports 71% holdout accuracy for a growth classifier on a sampled active-content set. The methodology on page 36 describes an 80/20 split but does not say it held out whole clients or future periods. The questions for my model are where its decline label comes from, whether any input crosses into the label window, and whether its score changes when clients are kept apart. The paper's sample and this warehouse analysis are different, so their accuracy numbers are not directly comparable.

## Load the May and June data

I keep the large daily records in DuckDB and bring only page-level results into pandas. The first run saves the results locally. The daily batches keep memory use down.


In [9]:
import getpass
import os
import tempfile
from datetime import date, timedelta
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

FEATURE_COLUMNS = (
    'log_prior30_impressions', 'log_prior30_clicks', 'prior30_ctr',
    'prior30_avg_position_filled', 'position_missing',
)

LABEL_END = date(2026, 6, 30)
HISTORICAL_CUTOFF = LABEL_END - timedelta(days=30)
FEATURE_START = HISTORICAL_CUTOFF - timedelta(days=29)
CURRENT_START = HISTORICAL_CUTOFF + timedelta(days=1)

def repo_root() -> Path:
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / 'work' / 'notebooks').is_dir():
            return path
    raise FileNotFoundError('Open the repository or a folder inside it before running the notebook')

def _quote_path(path: Path) -> str:
    return str(path).replace('\\', '/').replace("'", "''")

def _feature_select(table: str, labeled: bool) -> str:
    extra = (
        ", future30_impressions, CASE WHEN future30_impressions < 0.8 * prior30_impressions "
        "THEN 1 ELSE 0 END AS future_decline_label"
    ) if labeled else ''
    return f"""
        SELECT client_hash_id, content_hash_id, prior30_impressions, prior30_clicks,
               prior30_avg_position,
               LN(1 + prior30_impressions) AS log_prior30_impressions,
               LN(1 + prior30_clicks) AS log_prior30_clicks,
               CAST(prior30_clicks AS DOUBLE) / prior30_impressions AS prior30_ctr,
               COALESCE(prior30_avg_position, 0.0) AS prior30_avg_position_filled,
               CASE WHEN prior30_avg_position IS NULL THEN 1 ELSE 0 END AS position_missing
               {extra}
        FROM {table}
    """

repo = repo_root()


In [10]:
def _summarize_month(con: duckdb.DuckDBPyConnection, source: str,
                     start: date, end: date, folder: Path) -> Path:
    """Stream a month to disk, then deduplicate one day at a time."""
    folder.mkdir()
    stage = folder / 'daily_source.parquet'
    days = folder / 'days'
    days.mkdir()
    summary = folder / 'summary.parquet'
    con.execute(f"""
        COPY (SELECT report_date, client_hash_id, content_hash_id,
                     gsc_impressions, gsc_clicks, gsc_avg_position
              FROM {source}
              WHERE report_date BETWEEN DATE '{start}' AND DATE '{end}')
        TO '{_quote_path(stage)}' (FORMAT PARQUET)
    """)
    day = start
    while day <= end:
        day_path = days / f'{day}.parquet'
        con.execute(f"""
            COPY (
                SELECT client_hash_id, content_hash_id,
                       MIN(gsc_impressions) AS impressions,
                       MIN(gsc_clicks) AS clicks,
                       CASE WHEN MIN(gsc_avg_position) > 0 AND MIN(gsc_impressions) > 0
                            THEN MIN(gsc_avg_position) * MIN(gsc_impressions)
                            ELSE 0 END AS position_numerator,
                       CASE WHEN MIN(gsc_avg_position) > 0 AND MIN(gsc_impressions) > 0
                            THEN MIN(gsc_impressions) ELSE 0 END AS position_denominator,
                       COUNT(*) - 1 AS repeated_rows,
                       CASE WHEN MIN(gsc_impressions) IS DISTINCT FROM MAX(gsc_impressions)
                                      OR MIN(gsc_clicks) IS DISTINCT FROM MAX(gsc_clicks)
                                      OR MIN(gsc_avg_position) IS DISTINCT FROM MAX(gsc_avg_position)
                                      OR (COUNT(gsc_impressions) > 0 AND COUNT(gsc_impressions) < COUNT(*))
                                      OR (COUNT(gsc_clicks) > 0 AND COUNT(gsc_clicks) < COUNT(*))
                                      OR (COUNT(gsc_avg_position) > 0 AND COUNT(gsc_avg_position) < COUNT(*))
                            THEN 1 ELSE 0 END AS conflicting_keys
                FROM read_parquet('{_quote_path(stage)}')
                WHERE report_date = DATE '{day}'
                GROUP BY client_hash_id, content_hash_id
            ) TO '{_quote_path(day_path)}' (FORMAT PARQUET)
        """)
        day += timedelta(days=1)
    stage.unlink()
    con.execute(f"""
        COPY (
            SELECT client_hash_id, content_hash_id, COUNT(*) AS observed_days,
                   COUNT(*) FILTER (WHERE impressions IS NULL OR clicks IS NULL) AS missing_days,
                   SUM(impressions) AS impressions, SUM(clicks) AS clicks,
                   SUM(position_numerator) / NULLIF(SUM(position_denominator), 0) AS avg_position,
                   SUM(repeated_rows) AS repeated_rows,
                   SUM(conflicting_keys) AS conflicting_keys
            FROM read_parquet('{_quote_path(days / '*.parquet')}')
            GROUP BY client_hash_id, content_hash_id
        ) TO '{_quote_path(summary)}' (FORMAT PARQUET)
    """)
    for day_path in days.glob('*.parquet'):
        day_path.unlink()
    days.rmdir()
    return summary


In [11]:
def build_frames(source_override: str | None = None,
                 connection: duckdb.DuckDBPyConnection | None = None
                 ) -> tuple[pd.DataFrame, pd.DataFrame]:
    output = repo_root() / 'work' / 'outputs'
    output.mkdir(parents=True, exist_ok=True)
    con = connection or duckdb.connect()
    with tempfile.TemporaryDirectory(prefix='lane2_build_', dir=output) as scratch:
        temp = Path(scratch)
        spill = output / 'lane2_duckdb_spill'
        spill.mkdir(exist_ok=True)
        con.execute(f"SET temp_directory = '{_quote_path(spill)}'")
        con.execute("SET memory_limit = '1GB'")
        con.execute('SET threads = 2')
        if source_override is None:
            token = os.environ.get('HF_TOKEN') or getpass.getpass('Hugging Face READ token: ')
            con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '"
                        + token.replace("'", "''") + "')")
            del token
            root = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance'
            may_source = f"read_parquet('{root}/month=2026-05/*.parquet')"
            june_source = f"read_parquet('{root}/month=2026-06/*.parquet')"
        else:
            may_source = june_source = source_override

        may = _summarize_month(con, may_source, FEATURE_START, HISTORICAL_CUTOFF,
                               temp / 'may')
        june = _summarize_month(con, june_source, CURRENT_START, LABEL_END,
                                temp / 'june')
        may_rel = f"read_parquet('{_quote_path(may)}')"
        june_rel = f"read_parquet('{_quote_path(june)}')"
        counts = con.sql(f"""
            SELECT SUM(repeated_rows), SUM(conflicting_keys),
                   COUNT(*) FILTER (WHERE client_hash_id IS NULL OR content_hash_id IS NULL)
            FROM (SELECT * FROM {may_rel} UNION ALL SELECT * FROM {june_rel})
        """).fetchone()
        repeated, conflicting, missing_keys = (int(value or 0) for value in counts)
        assert missing_keys == 0, f'{missing_keys:,} rows have a missing daily key'
        assert conflicting == 0, f'{conflicting:,} daily keys have different GSC values'

        con.execute(f"""
            CREATE OR REPLACE TEMP TABLE historical_metrics AS
            SELECT m.client_hash_id, m.content_hash_id,
                   m.impressions AS prior30_impressions,
                   m.clicks AS prior30_clicks,
                   m.avg_position AS prior30_avg_position,
                   j.impressions AS future30_impressions
            FROM {may_rel} m
            JOIN {june_rel} j USING (client_hash_id, content_hash_id)
            WHERE m.observed_days = 30 AND m.missing_days = 0
              AND m.impressions >= 100
              AND j.observed_days = 30 AND j.missing_days = 0
        """)
        con.execute(f"""
            CREATE OR REPLACE TEMP TABLE current_metrics AS
            SELECT client_hash_id, content_hash_id,
                   impressions AS prior30_impressions,
                   clicks AS prior30_clicks,
                   avg_position AS prior30_avg_position
            FROM {june_rel}
            WHERE observed_days = 30 AND missing_days = 0 AND impressions >= 100
        """)
        historical = con.sql(_feature_select('historical_metrics', labeled=True)).df()
        current = con.sql(_feature_select('current_metrics', labeled=False)).df()

    assert len(historical) > 0 and len(current) > 0, 'No pages met the coverage and impression rules'
    assert historical[list(FEATURE_COLUMNS)].notna().all().all()
    assert current[list(FEATURE_COLUMNS)].notna().all().all()
    print(f'May-June repeated GSC rows removed: {repeated:,}')
    print(f'Historical labeled pages: {len(historical):,}; current June pages: {len(current):,}')
    return historical, current

def load_frames() -> tuple[pd.DataFrame, pd.DataFrame]:
    output = repo_root() / 'work' / 'outputs'
    history_path = output / 'lane2_history_2026-05-31.parquet'
    current_path = output / 'lane2_current_2026-06-30.parquet'
    if history_path.exists() and current_path.exists():
        con = duckdb.connect()
        history = con.sql(f"SELECT * FROM read_parquet('{_quote_path(history_path)}')").df()
        current = con.sql(f"SELECT * FROM read_parquet('{_quote_path(current_path)}')").df()
        print(f'Loaded cached frames: {len(history):,} historical, {len(current):,} current pages')
        return history, current
    history, current = build_frames()
    output.mkdir(parents=True, exist_ok=True)
    con = duckdb.connect()
    con.register('cache_history', history)
    con.register('cache_current', current)
    con.execute(f"COPY cache_history TO '{_quote_path(history_path)}' (FORMAT PARQUET)")
    con.execute(f"COPY cache_current TO '{_quote_path(current_path)}' (FORMAT PARQUET)")
    print('Saved local Parquet caches under work/outputs; dataset files are gitignored')
    return history, current


In [12]:
def grouped_split(frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    groups = frame['client_hash_id']
    if groups.nunique() < 2:
        raise ValueError('A grouped split needs at least two clients')
    train, test = next(GroupShuffleSplit(n_splits=1, test_size=0.25,
                                         random_state=42).split(frame, groups=groups))
    assert set(groups.iloc[train]).isdisjoint(set(groups.iloc[test]))
    return train, test

def precision_at_k(scores, labels, k: int) -> tuple[float, int]:
    values = np.asarray(scores, dtype=float)
    truth = np.asarray(labels, dtype=int)
    if len(values) != len(truth) or len(values) == 0:
        raise ValueError('Scores and labels need the same nonzero length')
    used = min(k, len(values))
    order = np.argsort(-values, kind='stable')[:used]
    return float(truth[order].mean()), used


In [13]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

history, current = load_frames()
position_rows = history.loc[history.prior30_avg_position > 0].copy()
position_rows['tier'] = pd.cut(position_rows.prior30_avg_position,
                               bins=[0, 3, 10, 20, 50, np.inf],
                               labels=['top 3', 'page 1', '11 to 20', '21 to 50', 'over 50'])
for tier, group in position_rows.groupby('tier', observed=True):
    if len(group) >= 50:
        weighted_ctr = group.prior30_clicks.sum() / group.prior30_impressions.sum()
        print(f'{tier}: {len(group):,} pages; weighted May CTR {weighted_ctr:.2%}')
    else:
        print(f'{tier}: {len(group):,} pages; too few for a stable comparison')
print('This is a separate warehouse slice, not a reproduction of the paper portfolio')

Loaded cached frames: 102,532 historical, 96,678 current pages
top 3: 789 pages; weighted May CTR 0.60%
page 1: 36,081 pages; weighted May CTR 0.37%
11 to 20: 29,061 pages; weighted May CTR 0.35%
21 to 50: 30,399 pages; weighted May CTR 0.20%
over 50: 6,202 pages; weighted May CTR 0.04%
This is a separate warehouse slice, not a reproduction of the paper portfolio


## 2. My model under two splits

The random split puts pages from the same client on both sides. The grouped split holds out entire clients and is the main result for this question. Both use the same five May features, the same June decline label, and the same logistic model. The label base rate is printed beside precision at 20 and 50. Neither split tests a later cutoff because July outcomes are not in this release.

In [14]:
features = list(FEATURE_COLUMNS)
X = history[features]
y = history.future_decline_label.astype(int)
random_train, random_test = train_test_split(
    np.arange(len(history)), test_size=0.25, random_state=42, stratify=y
)
group_train, group_test = grouped_split(history)

def check_split(name, train, test):
    fitted = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
    fitted.fit(X.iloc[train], y.iloc[train])
    scores = fitted.predict_proba(X.iloc[test])[:, 1]
    base_rate = y.iloc[test].mean()
    p20, n20 = precision_at_k(scores, y.iloc[test], 20)
    p50, n50 = precision_at_k(scores, y.iloc[test], 50)
    print(f'{name}: {len(test):,} test pages; decline base rate {base_rate:.1%}; '
          f'precision at {n20} {p20:.1%}; precision at {n50} {p50:.1%}')
    return fitted, scores, p20, n20, base_rate

random_model, random_scores, random_p20, random_k20, random_base = check_split(
    'Random page split', random_train, random_test
)
group_model, group_scores, group_p20, group_k20, group_base = check_split(
    'Held-out clients', group_train, group_test
)
assert set(history.client_hash_id.iloc[group_train]).isdisjoint(
    set(history.client_hash_id.iloc[group_test])
)
rule_visibility_min = history.prior30_impressions.iloc[group_train].median()
rule_low_ctr_max = history.prior30_ctr.iloc[group_train].quantile(0.25)
rule_test = history.iloc[group_test]
rule_visible = rule_test.prior30_impressions >= rule_visibility_min
rule_low_ctr = rule_test.prior30_ctr <= rule_low_ctr_max
rule_striking = rule_test.prior30_avg_position.between(11, 20, inclusive='both')
rule_scores = (rule_visible.astype(int) *
               (2 + rule_low_ctr.astype(int) + rule_striking.astype(int))
               + 0.01 * np.log1p(rule_test.prior30_impressions))
baseline_p20, baseline_k20 = precision_at_k(
    rule_scores, y.iloc[group_test], 20
)
print(f'Simple rule precision at {baseline_k20}: {baseline_p20:.1%}')

Random page split: 25,633 test pages; decline base rate 65.8%; precision at 20 80.0%; precision at 50 84.0%
Held-out clients: 10,763 test pages; decline base rate 65.5%; precision at 20 80.0%; precision at 50 82.0%
Simple rule precision at 20: 95.0%


## 3. Leakage audit

The allowed model inputs come from May. IDs are used only for grouping. June impressions and the derived decline label are kept out of the input list. Query-window values, GA4 fields, and product scores are also absent. A separate one-split experiment deliberately adds the June-to-May impression ratio; its score shows what happens when the answer is handed to a model. That leaky result is not a valid model result.

In [15]:
forbidden_words = ('future', 'label', 'hash', 'url', 'query', 'ga4', 'score', 'flag')
assert all(not any(word in name for word in forbidden_words) for name in features)
assert not X.isna().any().any()
leaky_X = X.assign(future_to_prior_ratio=(
    history.future30_impressions / history.prior30_impressions
))
leaky_tree = DecisionTreeClassifier(max_depth=1, random_state=42)
leaky_tree.fit(leaky_X.iloc[group_train], y.iloc[group_train])
leaky_accuracy = accuracy_score(
    y.iloc[group_test], leaky_tree.predict(leaky_X.iloc[group_test])
)
print('Allowed features:', ', '.join(features))
print(f'Deliberately leaky ratio accuracy on held-out clients: {leaky_accuracy:.1%}')
if leaky_accuracy < 0.99:
    print('The leakage sanity check needs inspection')
print('The deliberate ratio is not included in the real model')

Allowed features: log_prior30_impressions, log_prior30_clicks, prior30_ctr, prior30_avg_position_filled, position_missing
Deliberately leaky ratio accuracy on held-out clients: 100.0%
The deliberate ratio is not included in the real model


## 4. Claim rewrite

Saying the model finds pages that need a refresh goes beyond the evidence. I can report how May signals ranked pages with an observed June decline on held-out clients, alongside the simple rule and the decline base rate. The model did worse than the rule on this split. This supports a cautious review process, not a claim that a refresh changes traffic.


In [16]:
print('Earlier claim: The model identifies pages that need a refresh.')
print(f'Revised claim: At the 31 May 2026 cutoff, the model found observed June '
      f'declines in {group_p20:.1%} of its top {group_k20} held-out-client pages. '
      f'The simple rule found {baseline_p20:.1%}, and the decline base rate was '
      f'{group_base:.1%}. The model did not beat the rule on this split. '
      'Neither score measures the effect of refreshing a page.')


Earlier claim: The model identifies pages that need a refresh.
Revised claim: At the 31 May 2026 cutoff, the model found observed June declines in 80.0% of its top 20 held-out-client pages. The simple rule found 95.0%, and the decline base rate was 65.5%. The model did not beat the rule on this split. Neither score measures the effect of refreshing a page.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.